In [1]:
# Load packages
import os
import hail as hl
import pandas as pd 

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
        "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Loading BokehJS ...

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20260930 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migration-aou-rw-9f879fc4
wb-silky-artichoke-2408.C2024Q3R9



In [2]:
# Run this cell once
hl.init(
    gcs_requester_pays_configuration=os.environ['GOOGLE_CLOUD_PROJECT']
)

Running on Apache Spark version 3.5.3
SparkUI available at http://jupyterlabsparkclusterforaouspark20260930-m.us-west2-c.c.wb-fast-radish-6415.internal:37453
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.135-034ef3e08116
LOGGING: writing to /home/dataproc/workspace/rw-migration-aou-rw-9f879fc4/notebooks/hail/hail-20260930-2226-0.2.135-034ef3e08116.log
2026-09-30 22:27:15.573 Hail: WARN: entries(): Resulting entries table is sorted by '(row_key, col_key)'.
    To preserve row-major matrix table order, first unkey columns with 'key_cols_by()'
2026-09-30 22:48:18.307 Hail: INFO: merging 100 files totalling 32.5M...
2026-09-30 22:48:20.706 Hail: INFO: while writing:
    gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr17_76730000_76740000.tsv.bgz
  merge time: 2.398s


In [3]:
# Donwload exclusion samples 
!gcloud storage cp gs://vwb-aou-datasets-controlled/v8/known_issues/wgs_v8_known_issue_1.txt . --billing-project=${GOOGLE_CLOUD_PROJECT}

Copying gs://vwb-aou-datasets-controlled/v8/known_issues/wgs_v8_known_issue_1.txt to file://./wgs_v8_known_issue_1.txt
  Completed files 1/1 | 35.0B/35.0B                                            


In [4]:
# Exclusion samples 
exclusion = pd.read_csv('wgs_v8_known_issue_1.txt', sep='\t')
samples_to_exclude = exclusion['research_id'].values

In [5]:
# ============================================================
# Load MatrixTable
# ============================================================

mt_wgs_path = (
    "gs://vwb-aou-datasets-controlled/"
    "v8/wgs/short_read/snpindel/exome/splitMT/hail.mt"
)

mt = hl.read_matrix_table(mt_wgs_path)


# ============================================================
# Prepare exclusion samples ONCE
# ============================================================

samples_to_exclude = {
    str(x) for x in samples_to_exclude
}

print(f"Samples to exclude: {len(samples_to_exclude)}")


# ============================================================
# Regions
# ============================================================

regions = [
    "chr17:76730000-76740000"
]


# ============================================================
# Process regions
# ============================================================

for region_string in regions:

    print(f"\n{'=' * 70}")
    print(f"Processing {region_string}")
    print(f"{'=' * 70}")

    # --------------------------------------------------------
    # Define genomic interval
    # --------------------------------------------------------

    interval = hl.parse_locus_interval(
        region_string,
        reference_genome="GRCh38"
    )

    # --------------------------------------------------------
    # Filter MatrixTable to region
    # --------------------------------------------------------

    mt_region = hl.filter_intervals(
        mt,
        [interval]
    )

    # --------------------------------------------------------
    # Remove excluded samples
    # --------------------------------------------------------

    mt_region = mt_region.filter_cols(
        ~hl.literal(samples_to_exclude).contains(
            mt_region.s
        )
    )

    # --------------------------------------------------------
    # Convert to entry table
    # --------------------------------------------------------

    ht = mt_region.entries()

    # --------------------------------------------------------
    # Keep only called ALT genotypes
    # --------------------------------------------------------

    ht = ht.filter(
        hl.is_defined(ht.GT) &
        (ht.GT.n_alt_alleles() > 0)
    )

    # --------------------------------------------------------
    # Calculate DP
    # --------------------------------------------------------

    ht = ht.annotate(
        DP=hl.if_else(
            hl.is_defined(ht.AD),
            hl.sum(ht.AD),
            hl.missing(hl.tint32)
        )
    )

    # --------------------------------------------------------
    # Calculate VAF and AB
    # --------------------------------------------------------

    ht = ht.annotate(
        VAF=hl.if_else(
            hl.is_defined(ht.AD) &
            (ht.DP > 0),
            ht.AD[1] / ht.DP,
            hl.missing(hl.tfloat64)
        ),

        AB=hl.if_else(
            hl.is_defined(ht.AD) &
            (ht.DP > 0) &
            ht.GT.is_het(),
            ht.AD[1] / ht.DP,
            hl.missing(hl.tfloat64)
        )
    )

    # --------------------------------------------------------
    # Remove locus/s columns from table key
    #
    # entries() creates a key on locus + s.
    # This allows us to select locus and s as ordinary fields.
    # --------------------------------------------------------

    ht = ht.key_by()

    # --------------------------------------------------------
    # Keep required fields
    # --------------------------------------------------------

    ht = ht.select(
        "locus",
        "alleles",
        "s",
        "DP",
        "VAF",
        "AB",
        "GT",
        "AD",
        "GQ",
        "PS",
        "RGQ",
        "FT"
    )

    # --------------------------------------------------------
    # Increase partitions before export
    # --------------------------------------------------------

    print("Repartitioning entry table...")

    ht = ht.repartition(
        100,
        shuffle=True
    )

    # --------------------------------------------------------
    # Output filename
    # --------------------------------------------------------

    region_name = (
        region_string
        .replace(":", "_")
        .replace("-", "_")
    )

    output = (
        f'{os.environ["WORKSPACE_BUCKET"]}/data/hail/'
        f"variant_genotype_information_"
        f"{region_name}.tsv.bgz"
    )

    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    print(f"Exporting {output}...")

    ht.export(output)

    print(f"Finished {output}")

Samples to exclude: 3

Processing chr17:76730000-76740000
Repartitioning entry table...
Exporting gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr17_76730000_76740000.tsv.bgz...


Finished gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr17_76730000_76740000.tsv.bgz


In [6]:
!gcloud storage ls {os.environ["WORKSPACE_BUCKET"]}/data/hail/variant_genotype_information_{region_name}.tsv.bgz

gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr17_76730000_76740000.tsv.bgz


In [9]:
!gcloud storage cp {os.environ["WORKSPACE_BUCKET"]}/data/hail/variant_genotype_information_{region_name}.tsv.bgz .

Copying gs://rw-migration-aou-rw-9f879fc4/data/hail/variant_genotype_information_chr17_76730000_76740000.tsv.bgz to file://./variant_genotype_information_chr17_76730000_76740000.tsv.bgz
  Completed files 1/1 | 32.5MiB/32.5MiB                                        

Average throughput: 58.7MiB/s


In [13]:
import pandas as pd

file = "variant_genotype_information_chr17_76730000_76740000.tsv.bgz"

df = pd.read_csv(
    file,
    sep="\t",
    compression="gzip"
)

df.head()

,locus,alleles,s,DP,VAF,AB,GT,AD,GQ,PS,RGQ,FT
0,chr17:76732966,"[""A"",""G""]",3260122,22,0.50000,0.50000,0/1,"[11,11]",48,NaN,85,PASS
1,chr17:76732986,"[""T"",""C""]",2488907,37,0.51351,0.51351,0/1,"[18,19]",48,NaN,85,PASS
2,chr17:76733028,"[""A"",""G""]",2829394,30,0.60000,0.60000,0/1,"[12,18]",47,NaN,85,PASS
3,chr17:76733029,"[""G"",""C""]",1050136,37,0.48649,0.48649,0/1,"[19,18]",48,NaN,85,PASS
4,chr17:76733029,"[""G"",""C""]",1098112,35,0.68571,0.68571,0/1,"[11,24]",42,NaN,85,PASS
